[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# Why Pandas


## What you will be able to do

Load a year of weather readings into a pandas table and ask it questions by name, so that every
answer arrives labeled with the station and the hour it belongs to.

## The idea

### The problem

The **NumPy, Deep Dive** guide kept a year of hourly readings in one array of shape (4, 8760). Row 1
was Oslo and column 4,000 was an afternoon in June, and none of that was written anywhere in the
array. The code had to remember it. A mean for each station came back as four bare numbers in the
order of the rows, and turning a column number into a date needed a helper function of its own.

That arrangement holds up while one person writes the code and the stations never change. It breaks
when the data arrives from somewhere else. A second file lists the stations' latitudes in its own
order, so lining it up with the readings means sorting both by hand and hoping nothing was missing.
A fifth station joins the network, and every position after it moves. Each of those mistakes gives a
wrong answer and no error, because to NumPy a position is a position.

Most data you will meet comes as a table, too: a CSV export, a spreadsheet, the result of a database
query. Each column has a name, the columns hold different types, and a row is one observation, such
as one station at one hour. An array of a single type fits that shape badly.

### What a DataFrame is

> A **DataFrame** is a table whose columns each hold values of one type and share one **index**,
> the labels of its rows. A single column is a **Series**, a NumPy array with that index attached.
> Operations between tables and columns line up their values by label before they compute.

### Why it works that way

A label travels with its value. Ask for each station's mean and the answer is a Series indexed by
station name, so Svalbard's mean is next to the word Svalbard however the result is sorted or
filtered later. A table of latitudes combined with that Series lines up by station name, and a
station that appears in only one of them gets a missing value in the result instead of somebody
else's number.

Underneath, each column is still a NumPy array, so the arithmetic is vectorized as in the NumPy
guide. pandas adds the labels and a set of operations that use them: filtering rows by a condition
on a column, grouping rows that share a value, joining tables on a key. It also makes a different
default choice about missing values. A mean in pandas leaves `NaN` out, where NumPy's `.mean()`
returns `NaN`.

The readings in this guide are stored long, one row for each station at each hour, 35,040 rows in
all. The **Reshaping** notebook turns them wide, with a column for each station, and explains when
each shape is the easier one to ask questions of.

### Where this shows up

pandas is where most data analysis in Python happens. A CSV file loads into a DataFrame in one line, a database query
returns one, and Matplotlib and scikit-learn both accept one. The **CSV** notebook in the **Files,
Paths and Formats** guide read a file with the `csv` module, row by row. This guide reads files with
`read_csv`, and the readings are the ones the **sqlite3, Deep Dive** and **NumPy, Deep Dive** guides
use, made by the same formula, so their numbers and this guide's agree hour by hour.

### What this notebook covers

- the readings as a table, and one column as a Series
- the same question asked by position in an array and by name in a table
- each station's mean, warmest hour and monthly means, labeled
- two tables lined up by station name, including a station with no readings
- three errors, among them a sort that puts the missing readings where the warmest ones should be

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import pandas as pd

readings = pd.DataFrame({
    "station": ["Bergen", "Oslo", "Bergen", "Oslo"],
    "celsius": [8.2, 6.1, 7.9, 5.8],
})
print(readings.groupby("station")["celsius"].mean())
```

```
station
Bergen    8.05
Oslo      5.95
Name: celsius, dtype: float64
```

The table has two columns and four rows. `groupby("station")` put the rows of each station together,
`["celsius"]` chose the column to work on, and `.mean()` averaged each group. The answer names its
stations, so nothing about the order of the rows had to be remembered.

## Setup

Two libraries, three helpers for building the readings, and two tables.

- `pandas`, imported as `pd` as nearly every program that uses it does, is the library, and the cell
  prints its version
- `numpy`, imported as `np`, appears once, for the comparison with the NumPy guide's array in the
  first worked examples
- `math`, `datetime` and `timedelta` build the readings

`year_of_readings` makes a reading for every hour of 2025 at the four stations in `STATIONS`, by the
same formula as the **sqlite3, Deep Dive** guide, and Svalbard sent nothing on 2 March, so those 24
readings are `None`. `readings` is the table of all 35,040, with the columns `station`, `time` and
`celsius`. `pd.to_datetime` turns the text in `time` into dates and times, and `None` became `NaN` in
`celsius`. `stations` holds each station's latitude from `LATITUDES`, and adds Kirkenes, a station
with no readings.

In [1]:
import math
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})

print("pandas", pd.__version__, "|", len(readings), "readings,", len(stations), "stations")

pandas 2.2.3 | 35040 readings, 5 stations


## Worked examples

### The readings as a table

A DataFrame that ends a cell shows as a table, and a table with more rows than fit shows its first
five and last five, with its size underneath:

In [2]:
readings

,station,time,celsius
0,Bergen,2025-01-01 00:00:00,-3.6
1,Oslo,2025-01-01 00:00:00,-3.5
2,Svalbard,2025-01-01 00:00:00,-14.6
3,Tromso,2025-01-01 00:00:00,-6.7
4,Bergen,2025-01-01 01:00:00,-3.7
...,...,...,...
35035,Tromso,2025-12-31 22:00:00,-6.1
35036,Bergen,2025-12-31 23:00:00,-1.7
35037,Oslo,2025-12-31 23:00:00,-3.3
35038,Svalbard,2025-12-31 23:00:00,-14.4


The column of numbers on the left is the index. Nobody chose it, so pandas numbered the rows from 0,
and the **Series** notebook comes back to that. Each row is one station at one hour, which is why
Bergen, Oslo, Svalbard and Tromso take turns in the first four rows. `time` shows dates and times,
because Setup parsed it, and the last hour of the year is 23:00 on 31 December.

### A single column as a Series

Square brackets with a column's name return that column:

In [3]:
celsius = readings["celsius"]

print(type(celsius).__name__, celsius.dtype, len(celsius))
print(celsius.head(4))
print("mean of every reading:", round(celsius.mean(), 2))

Series float64 35040
0    -3.6
1    -3.5
2   -14.6
3    -6.7
Name: celsius, dtype: float64
mean of every reading: 3.38


The Series has the same index as the table, 0 to 35039, and holds `float64` values, the decimals of
the NumPy guide. `.mean()` averaged all 35,040 readings and left out the 24 that are `NaN`.

### The same question by position and by name

The NumPy guide would answer "what was Oslo's mean?" with an array of the readings. Here are the same
values in that shape, with one column for each station, and Oslo's mean taken from column 1:

In [4]:
values = readings["celsius"].to_numpy().reshape(8760, 4)     # an hour per row, a station per column

print(values.shape)
print("Oslo, by position:", round(np.nanmean(values[:, 1]), 2))

(8760, 4)
Oslo, by position: 6.5


That answer is right only because the readings happen to cycle through the stations in the order of
`STATIONS`, so column 1 is Oslo. Nothing in `values` says so. The table can be asked by name:

In [5]:
oslo = readings[readings["station"] == "Oslo"]

print("Oslo, by name:", round(oslo["celsius"].mean(), 2))
print(len(oslo), "readings")

Oslo, by name: 6.5
8760 readings


`readings["station"] == "Oslo"` is a Series of `True` and `False`, one for every row, and putting it
in square brackets keeps the rows where it is `True`. The **Boolean Filtering** notebook covers it.
The question names the station, and it would still name the right one if the rows came in another
order.

### A labeled mean for each station

In [6]:
means = readings.groupby("station")["celsius"].mean().round(2)
means

station
Bergen      8.00
Oslo        6.50
Svalbard   -4.48
Tromso      3.50
Name: celsius, dtype: float64

The answer is a Series whose index is the station names. Svalbard's mean is a number, -4.48, and not
`NaN`, because a pandas mean leaves missing values out. The **groupby** notebook takes `groupby` apart.

### The warmest hour at each station

`idxmax` returns the index label of each group's largest value, and `loc` turns those labels back into
rows of the table:

In [7]:
warmest = readings.groupby("station")["celsius"].idxmax()
readings.loc[warmest]

,station,time,celsius
19356,Bergen,2025-07-21 15:00:00,20.8
18973,Oslo,2025-07-17 15:00:00,19.3
18590,Svalbard,2025-07-13 15:00:00,8.3
18207,Tromso,2025-07-09 15:00:00,16.2


Each row keeps its station, its time and its reading together, so there is nothing to convert. All
four peaks fall in July, at 15:00, the hour the formula's daily swing is at its highest. The **loc and
iloc** notebook covers `loc`.

### Monthly means as a table

Grouping by two things, the station and the month of `time`, gives one mean for each pair, and
`unstack`, which the **Reshaping** notebook explains, spreads the months across the columns:

In [8]:
monthly = readings.groupby(["station", readings["time"].dt.month])["celsius"].mean().unstack().round(1)
monthly

time,1,2,3,4,5,6,7,8,9,10,11,12
station,,,,,,,,,,,,
Bergen,-0.9,0.1,3.2,7.5,12.1,15.5,16.9,15.9,12.7,8.3,3.8,0.4
Oslo,-2.4,-1.4,1.7,6.0,10.6,14.0,15.4,14.4,11.2,6.8,2.3,-1.1
Svalbard,-13.4,-12.4,-9.3,-5.0,-0.4,3.0,4.4,3.4,0.2,-4.2,-8.7,-12.1
Tromso,-5.4,-4.4,-1.3,3.0,7.6,11.0,12.4,11.4,8.2,3.8,-0.7,-4.1


`readings["time"].dt.month` is the month of every reading, from 1 to 12, which the **Dates and Times**
notebook covers. The table has a row for each station and a column for each month, and Svalbard's
July, the warmest month there, is 4.4 degrees.

### Lining up tables by station name

The latitudes come from a second table, `stations`, in an order of their own. Here they are sorted
from south to north, which is a different order from `means`, and with Kirkenes, which has no
readings:

In [9]:
latitude = stations.set_index("station")["latitude"].sort_values()
latitude

station
Oslo        59.91
Bergen      60.39
Tromso      69.65
Kirkenes    69.73
Svalbard    78.22
Name: latitude, dtype: float64

`set_index("station")` made the station names the index, which **The Index** notebook covers.
Putting the two Series side by side in one table lines them up by label:

In [10]:
pd.DataFrame({"mean": means, "latitude": latitude})

,mean,latitude
station,,
Bergen,8.00,60.39
Kirkenes,NaN,69.73
Oslo,6.50,59.91
Svalbard,-4.48,78.22
Tromso,3.50,69.65


Every station's mean sits beside its own latitude, although the two Series were in different orders.
Kirkenes is in `latitude` and not in `means`, so its mean is `NaN`. With two arrays, the same table
would have needed both lists sorted the same way first, and Kirkenes would have shifted every row
after it. The **Merging** notebook joins whole tables on a key column.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/01-why-pandas-solutions.ipynb).

**1.** Print Tromso's mean for the year, rounded to two places, by filtering `readings` on the station's name.

In [11]:
# your code here

**2.** Show the coldest hour at each station, with its time, using `idxmin`.

In [12]:
# your code here

**3.** Print each station's number of readings, with `readings.groupby("station")["celsius"].count()`, and say which station has fewer and why.

In [13]:
# your code here

**4.** From `monthly`, print each station's coldest monthly mean, with `monthly.min(axis=1)`.

In [14]:
# your code here

**5.** Make a table with each station's warmest reading of the year beside its latitude, the way the last worked example lined up the means.

In [15]:
# your code here

## Common errors

### KeyError: 'Celsius'

In [16]:
readings["Celsius"]

KeyError: 'Celsius'

Column names are labels, and labels are matched exactly, case included. The column is `celsius`.
`readings.columns` lists the names when you are not sure of one.

### No error, and the warmest hours are missing readings

In [17]:
readings.sort_values("celsius").tail(3)

,station,time,celsius
5846,Svalbard,2025-03-02 21:00:00,NaN
5850,Svalbard,2025-03-02 22:00:00,NaN
5854,Svalbard,2025-03-02 23:00:00,NaN


`sort_values` sorts from lowest to highest, so the warmest readings should be at the end. Missing
values go last by default, though, so Svalbard's 24 silent hours are the last 24 rows. `tail(3)` returned three of them. `nlargest` asks for the largest values and leaves `NaN` out:

In [18]:
readings.nlargest(3, "celsius")

,station,time,celsius
19356,Bergen,2025-07-21 15:00:00,20.8
18880,Bergen,2025-07-16 16:00:00,20.7
18108,Bergen,2025-07-08 15:00:00,20.6


Two of Bergen's hours tie at 20.6, and `nlargest` keeps the first of them in the table's order, so the
result is the same on every run.

### TypeError: agg function failed [how->mean,dtype->object]

In [19]:
readings.groupby(readings["time"].dt.month).mean()

TypeError: agg function failed [how->mean,dtype->object]

Grouping by month leaves `station` among the columns to average, and a column of names has no mean,
so the whole call raises. Choose the column to average before calling `.mean()`:

In [20]:
readings.groupby(readings["time"].dt.month)["celsius"].mean().round(2).head(3)

time
1   -5.52
2   -4.50
3   -1.39
Name: celsius, dtype: float64

## Recap

- Build a DataFrame and read one of its columns as a Series by name
- Ask a question by label, such as Oslo's mean, instead of by a position you have to remember
- Get a labeled answer for every station with `groupby`, and find each group's warmest row with
  `idxmax` and `loc`
- Line up two Series by label, and read `NaN` where one side has no value
- Find the largest values with `nlargest`, which leaves missing values out

## What is next

The **Series** notebook looks at one column on its own: its values, its index, and what lining up by
label does when two indexes disagree.

---

[Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)
